# IHARQ Phase 02 / Layer 02 — Complete Execution and Analysis R4

**Notebook ID:** `IHARQ-P02-COMPLETE-EXECUTION-AND-ANALYSIS-R4`  
**Build Book:** `IHARQ-P02-L2-INTEGRATED-BUILD-BOOK-R4`  
**Scientific freeze:** `P02-PLANNED-SCIENTIFIC-EXECUTION-FREEZE-R5`, created by owner-authorized pre-execution training-policy amendment `P02-PREEXEC-TRAINING-POLICY-AMENDMENT-R2`.

> **CURRENT PRE-EXECUTION STATUS — READY SUBJECT TO NORMAL RUNTIME GATES.** The former training-policy blockers are resolved by predeclared **validation-only resolution algorithms**, not by notebook-local magic numbers. P00/P01 are unchanged. The official A0/A4 surface remains exactly 1,896 cells. The separate EEGNet FULL_TRAIN S&R diagnostic surface remains 15 final dataset×seed cells outside A-number accounting. Stage 05 verifies the amendment before science; Stage 11 resolves the dataset-specific S&R probability and per-run class-weight policy using legal training/validation evidence only; no test evidence may select either policy.

This remains the **single full-scope P02 Kaggle notebook**. No P02 scientific execution has occurred and no synthetic fixture is P02 evidence.


In [ ]:
# Portable Kaggle bootstrap — standard library only until frozen dependencies are verified.
from pathlib import Path
import os, sys, shutil, subprocess, hashlib, json, importlib.metadata as _imd

NOTEBOOK_ID = "IHARQ-P02-COMPLETE-EXECUTION-AND-ANALYSIS-R4"
EXPECTED_PACKAGE_NAME = "IHARQ_P02_L2_Kaggle_Notebook_Implementation_Package_R5"
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
if not KAGGLE_INPUT.exists() or not KAGGLE_WORKING.exists():
    raise RuntimeError("KAGGLE_RUNTIME_REQUIRED")

def _candidate_roots():
    roots=[]
    for p in KAGGLE_INPUT.rglob("p02_notebook_stage_plan_R4.yaml"):
        if p.parent.name=="machine_readable":
            q=p.parent.parent
            if (q/"src/iharq/layer2_decoders").is_dir():
                roots.append(q)
    return roots

roots=_candidate_roots()
if len(roots)>1:
    raise RuntimeError(f"P02_IMPLEMENTATION_PACKAGE_EXPECTED_ONE:{len(roots)}")
if roots:
    PACKAGE_INPUT_ROOT=roots[0]
else:
    zips=list(KAGGLE_INPUT.rglob(EXPECTED_PACKAGE_NAME+".zip"))
    if len(zips)!=1:
        raise RuntimeError(f"P02_IMPLEMENTATION_PACKAGE_NOT_RESOLVED: extracted={len(roots)} zip={len(zips)}")
    import zipfile
    extract_root=KAGGLE_WORKING/"_p02_package_extract"
    if extract_root.exists(): shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(zips[0]) as z:
        bad=z.testzip()
        if bad is not None: raise RuntimeError(f"P02_PACKAGE_ZIP_CRC_FAIL:{bad}")
        if any(Path(n).is_absolute() or ".." in Path(n).parts for n in z.namelist()):
            raise RuntimeError("P02_PACKAGE_UNSAFE_ZIP_PATH")
        z.extractall(extract_root)
    matches=list(extract_root.rglob("p02_notebook_stage_plan_R4.yaml"))
    roots=[p.parent.parent for p in matches if p.parent.name=="machine_readable" and (p.parent.parent/"src/iharq/layer2_decoders").is_dir()]
    if len(roots)!=1: raise RuntimeError(f"P02_EXTRACTED_PACKAGE_EXPECTED_ONE:{len(roots)}")
    PACKAGE_INPUT_ROOT=roots[0]

PACKAGE_ROOT=KAGGLE_WORKING/EXPECTED_PACKAGE_NAME
if PACKAGE_ROOT.exists(): shutil.rmtree(PACKAGE_ROOT)
shutil.copytree(PACKAGE_INPUT_ROOT,PACKAGE_ROOT)

# Verify internal authoring-package checksum surface before importing project code.
def _sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""): h.update(chunk)
    return h.hexdigest()

checksum_file=PACKAGE_ROOT/"checksums.sha256"
if not checksum_file.is_file(): raise RuntimeError("P02_PACKAGE_CHECKSUM_MANIFEST_MISSING")
bad=[]
for line in checksum_file.read_text().splitlines():
    if not line.strip(): continue
    expected,rel=line.split("  ",1)
    p=PACKAGE_ROOT/rel
    if not p.is_file() or _sha(p)!=expected: bad.append(rel)
if bad: raise RuntimeError(f"P02_PACKAGE_INTERNAL_CHECKSUM_FAILURE:{bad[:5]}")

# Verify the owner-authorized pre-execution training-policy amendment before dependency installation or science.
_resolution_path=PACKAGE_ROOT/"validation/P02_FREEZE_CRITICAL_BLOCKER_REGISTER_R4.json"
if not _resolution_path.is_file():
    raise RuntimeError("P02_PREEXECUTION_RESOLUTION_REGISTER_MISSING")
_resolution_state=json.loads(_resolution_path.read_text())
if _resolution_state.get("status")!="PASS" or _resolution_state.get("freeze_critical_blocker_count")!=0:
    raise RuntimeError("P02_PREEXECUTION_SCIENTIFIC_FREEZE_NOT_RESOLVED")
_amendment_path=PACKAGE_ROOT/"contracts/P02_PREEXECUTION_TRAINING_POLICY_AMENDMENT_R2.yaml"
if not _amendment_path.is_file():
    raise RuntimeError("P02_PREEXECUTION_TRAINING_POLICY_AMENDMENT_MISSING")


# Frozen dependency intent. Install only if the exact set is not already present.
req=PACKAGE_ROOT/"requirements-kaggle.txt"
mismatches=[]
for line in req.read_text().splitlines():
    if "==" not in line or line.lstrip().startswith("#"): continue
    name,expected=line.split("==",1)
    try: actual=_imd.version(name)
    except _imd.PackageNotFoundError: actual=None
    if actual!=expected: mismatches.append((name,expected,actual))
if mismatches:
    subprocess.check_call([sys.executable,"-m","pip","install","--no-input","-r",str(req)])

src=PACKAGE_ROOT/"src"
sys.path.insert(0,str(src))
from iharq.layer2_decoders.identity import sha256_file
from iharq.layer2_decoders.kaggle_entry import production_session

def _tree_sha(root, suffixes):
    h=hashlib.sha256()
    for p in sorted(x for x in Path(root).rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(root).as_posix().encode()); h.update(b"\0"); h.update(p.read_bytes()); h.update(b"\0")
    return h.hexdigest()

REVISION={
    "notebook_revision":"R4",
    "notebook_id":NOTEBOOK_ID,
    "source_sha256":_tree_sha(PACKAGE_ROOT/"src",{".py"}),
    "config_sha256":_tree_sha(PACKAGE_ROOT/"configs",{".yaml",".yml"}),
    "stage_plan_sha256":sha256_file(PACKAGE_ROOT/"machine_readable/p02_notebook_stage_plan_R4.yaml"),
    "scientific_freeze_sha256":sha256_file(PACKAGE_ROOT/"machine_readable/p02_planned_scientific_execution_freeze_R5.yaml"),
    "scientific_freeze_id":"P02-PLANNED-SCIENTIFIC-EXECUTION-FREEZE-R5",
}
RUN_ROOT=KAGGLE_WORKING/"iharq_p02_run"
RUN_ROOT.mkdir(parents=True,exist_ok=True)
SESSION=production_session(PACKAGE_ROOT,RUN_ROOT,REVISION)
print(json.dumps({"status":"BOOTSTRAP_READY","package_root":str(PACKAGE_ROOT),"run_root":str(RUN_ROOT),"revision":REVISION},indent=2))


## Stage 00 — Identity, authorities, cumulative project-state intake

**Gate:** `G00`  
**Inputs:** cumulative ZIP  
**Outputs:** authority intake + run identity

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("00")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G00"},indent=2))

## Stage 01 — Environment/dependency setup and measured resource inventory

**Gate:** `G01`  
**Inputs:** environment plan  
**Outputs:** environment report

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("01")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G01"},indent=2))

## Stage 02 — Cumulative ZIP, manifest and checksum preflight

**Gate:** `G02`  
**Inputs:** project ZIP  
**Outputs:** verified intake ledger

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("02")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G02"},indent=2))

## Hugging Face conditional-asset authentication / retrieval preflight

This cell configures **credential transport only**; it does not authorize a model scientifically. The notebook first checks a private Kaggle Secret named `HF_TOKEN`. If none is available and this is an interactive session, it asks for a Hugging Face access token using a hidden prompt. Press **Enter** to continue without a token when you only need public eligible assets.

- **CBraMod:** public, immutable Hugging Face checkpoint; token optional.
- **REVE:** gated on Hugging Face, but **current P02 keeps it blocked before download because of known pretraining-corpus overlap with P02 benchmark sources**. A token cannot bypass that scientific gate.
- Token values remain in memory only and are never written to project configuration, logs, manifests, or the execution bundle.
- For unattended Kaggle **Save & Run All**, configure `HF_TOKEN` in Kaggle Secrets first; a background commit cannot answer an interactive prompt.


In [ ]:
# Secure Hugging Face token acquisition — no credential is hard-coded or persisted by IHARQ.
import os, getpass
HF_TOKEN = None
HF_TOKEN_SOURCE = "NONE"

# 1) Kaggle Secret: best choice for unattended Save & Run All.
try:
    from kaggle_secrets import UserSecretsClient
    _usc = UserSecretsClient()
    try:
        HF_TOKEN = (_usc.get_secret("HF_TOKEN") or "").strip() or None
        if HF_TOKEN:
            HF_TOKEN_SOURCE = "KAGGLE_SECRET:HF_TOKEN"
    except Exception:
        pass
except Exception:
    pass

# 2) Existing environment secret, if the runtime owner supplied one.
if not HF_TOKEN:
    HF_TOKEN = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip() or None
    if HF_TOKEN:
        HF_TOKEN_SOURCE = "ENVIRONMENT_SECRET_PROVIDER"

# 3) Hidden interactive fallback. Blank is valid for public CBraMod.
if not HF_TOKEN:
    try:
        _entered = getpass.getpass(
            "Hugging Face read/fine-grained token (hidden; optional for public CBraMod; press Enter to skip): "
        ).strip()
        HF_TOKEN = _entered or None
        if HF_TOKEN:
            HF_TOKEN_SOURCE = "INTERACTIVE_GETPASS_MEMORY_ONLY"
    except (EOFError, KeyboardInterrupt):
        HF_TOKEN = None
        HF_TOKEN_SOURCE = "NONINTERACTIVE_NO_SECRET"

HF_AUTH = SESSION.set_hf_token(HF_TOKEN, HF_TOKEN_SOURCE)
# Never print the token itself.
print(json.dumps({
    "huggingface_credential_available": bool(HF_TOKEN),
    "credential_source_type": HF_TOKEN_SOURCE,
    "credential_value": "REDACTED" if HF_TOKEN else "NOT_SET",
    "public_cbramod_download_can_continue_without_token": True,
    "reve_current_p02_status": "BLOCKED_KNOWN_PRETRAINING_CORPUS_OVERLAP_BEFORE_AUTH_DOWNLOAD"
}, indent=2))
try:
    del _entered
except NameError:
    pass


## Stage 03 — Resolve P01 core/A4 inputs + governed conditional Hugging Face assets

**Gate:** `G03`  
**Inputs:** pointer records  
**Outputs:** resolved immutable artifact map

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("03")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G03"},indent=2))
# Stage 03 is the only governed Hugging Face retrieval window. Clear the credential immediately afterwards.
_HF_CLEAR = SESSION.clear_hf_token()
HF_TOKEN = None
try:
    del _usc
except NameError:
    pass
print(json.dumps({"huggingface_credential_retained_after_stage03": False, "credential_source_state": _HF_CLEAR["source"]}, indent=2))


## Stage 04 — Validate P01 input contract; no relabel/resplit/rewindow

**Gate:** `G04`  
**Inputs:** P01 records/shards  
**Outputs:** input validation report

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("04")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G04"},indent=2))

## GOVERNANCE NOTE — FUTURE PROTOCOL / BUILD-BOOK SYNCHRONIZATION REQUIRED

The two former pre-execution blockers are resolved for this run by `P02-PREEXEC-TRAINING-POLICY-AMENDMENT-R2`. This is a **minimal owner-authorized pre-run algorithmic amendment**, not a retroactive rewrite of P00/P01.

The amendment does **not** predeclare arbitrary realized values when the run can lawfully determine them. Instead it freezes the resolution procedure before test access:

1. **EEGNet S&R diagnostic challenger:** primary EEGNet remains unaugmented. For each dataset, S&R application probability is selected before test inference from the compact frozen candidate set `{0.25, 0.50, 0.75}` using all five frozen EEGNet seeds and legal validation evidence: median validation BACC → median macro-F1 → proximity to 0.5 → lower probability. `n_segments` is resolved by the pinned Braindecode `SegmentationReconstruction(..., n_segments=None, ...)` public API and recorded. The final challenger remains exactly 15 dataset×seed cells, diagnostic-only, outside A0/A4 and ineligible for A4 representative selection/P03-primary status.
2. **Class weighting:** exactly equal fit-role class counts use uniform loss without an extra fit. If counts are unequal, the standard training-label-derived balanced vector `n/(K*n_c)` is computed and a weighted vs unweighted fit with the same selected model hyperparameters/seed is compared on validation BACC → macro-F1 → unweighted tie-break. Only the validation-selected policy reaches test. This applies only where the selected classifier/loss has native or explicitly verified class-weight support; no unsupported algorithm is silently rewritten.

Test labels/scores may never choose S&R probability, segment count, class weighting, model hyperparameters, checkpoint, or model family.

After the accepted P02 run, update the cumulative Protocol v1.0 and P02 Build Book successor using `docs/P02_FUTURE_PROTOCOL_BUILD_BOOK_SYNC_NOTE_R2.md`: incorporate the algorithmic policy and record the realized per-dataset/per-run execution bindings. Do not retune those policies from test results.

These changes do **not** alter P00/P01, A0=678, A4=1,218, A14 prohibition, the eight L2 modules, record schemas, expected figures/tables, P03 raw-prediction contract, or any downstream handoff.


In [ ]:
# Verify and display the exact pre-execution amendment identity before Stage 05.
import yaml, hashlib, json
_amendment_path = PACKAGE_ROOT / 'contracts' / 'P02_PREEXECUTION_TRAINING_POLICY_AMENDMENT_R2.yaml'
_amendment = yaml.safe_load(_amendment_path.read_text())
assert _amendment['status'] == 'FROZEN_ALGORITHM_RESOLVED_BY_OWNER_AUTHORIZED_PREEXECUTION_AMENDMENT_R2'
assert _amendment['amendment_id'] == 'P02-PREEXEC-TRAINING-POLICY-AMENDMENT-R2'
assert _amendment['freeze_critical_blockers'] == []
assert _amendment['official_A0_A4_cell_count_unchanged'] == 1896
assert _amendment['augmentation_challenger']['run_cell_count'] == 15
assert _amendment['augmentation_challenger']['probability_resolution']['test_set_access'] == 'PROHIBITED'
assert _amendment['class_weighting']['selection']['test_set_access'] == 'PROHIBITED'
_amendment_sha256 = hashlib.sha256(_amendment_path.read_bytes()).hexdigest()
print(json.dumps({
    'amendment_id': _amendment['amendment_id'],
    'sha256': _amendment_sha256,
    'resolution_type': 'PREDECLARED_TRAIN_VALIDATION_ONLY_ALGORITHMS',
    'future_protocol_build_book_sync_required': True,
    'P00_P01_mutated': False,
    'official_A0_A4_cells': 1896,
    'test_set_policy_selection': 'PROHIBITED',
}, indent=2))


## Stage 05 — Verify immutable P02 R5 scientific execution freeze, dynamic pre-test training-policy amendment, run-family expansion, full-ablation ownership/completeness matrix and one-notebook dispatcher

**Gate:** `G05`  
**Inputs:** Build Book R4 + `p02_planned_scientific_execution_freeze_R5.yaml` + `P02_PREEXECUTION_TRAINING_POLICY_AMENDMENT_R2.yaml` + 1,896 A0/A4 manifest + 15-cell R2 training-policy challenger manifest  
**Outputs:** verified config hash + run matrices + training-policy resolution contract + no-test-access invariants

Stage 05 does **not** choose realized policy values. It verifies the predeclared algorithms and their legal evidence roles. Later model fitting may resolve S&R probability and class-weight activation only from training/validation evidence under the frozen rules; test access is prohibited until those choices are fixed.


In [ ]:
result = SESSION.run("05")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G05"},indent=2))

## Stage 06 — Schema successor/profile, fixture, import and smoke validation

**Gate:** `G06`  
**Inputs:** schemas/code  
**Outputs:** schema validation + smoke evidence

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("06")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G06"},indent=2))

## Stage 07 — Governed data loaders/cache and role firewall

**Gate:** `G07`  
**Inputs:** HDF5 + manifests  
**Outputs:** loader/cache evidence

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("07")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G07"},indent=2))

## Stage 08 — Sanity controls

**Gate:** `G08`  
**Inputs:** core inputs  
**Outputs:** sanity models/logs

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("08")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G08"},indent=2))

## Stage 09 — Classical anchors

**Gate:** `G09`  
**Inputs:** core inputs  
**Outputs:** CSP/FBCSP checkpoints/logs

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("09")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G09"},indent=2))

## Stage 10 — Riemannian anchors/challengers/diagnostics

**Gate:** `G10`  
**Inputs:** core inputs  
**Outputs:** TS/EA/MDM artifacts

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("10")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G10"},indent=2))

## Stage 11 — Compact neural baseline

**Gate:** `G11`  
**Inputs:** core inputs  
**Outputs:** EEGNet + eligible neural checkpoints

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("11")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G11"},indent=2))

## Stage 12 — Conditional deep/SSL admission and execution

**Gate:** `G12`  
**Inputs:** gates/checkpoints  
**Outputs:** admitted branch artifacts or explicit block

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("12")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G12"},indent=2))

## Stage 13 — Checkpoint round-trip + ModelRegistry closure

**Gate:** `G13`  
**Inputs:** all branch artifacts  
**Outputs:** ModelRegistryRecord + reload evidence

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("13")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G13"},indent=2))

## Stage 14 — Governed inference and PredictionRecord generation

**Gate:** `G14`  
**Inputs:** accepted checkpoints + windows  
**Outputs:** PredictionRecord partitions

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("14")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G14"},indent=2))

## Stage 15 — A0 FULL EXECUTION/EVALUATION: terminal coverage, raw accept-all metrics, participant source tables, uncertainty/comparison evidence, figure/table/Analysis export

**Gate:** `G15`  
**Inputs:** PredictionRecords  
**Outputs:** A0 metric/source artifacts

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("15")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G15"},indent=2))

## Stage 16 — Low-label/low-calibration curves

**Gate:** `G16`  
**Inputs:** budgeted records  
**Outputs:** LowCalibrationCurveRecord

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("16")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G16"},indent=2))

## Stage 17 — Subject/session descriptive profiles

**Gate:** `G17`  
**Inputs:** prediction/metric records  
**Outputs:** SubjectProfileRecord

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("17")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G17"},indent=2))

## Stage 18 — A4 FULL EXECUTION/EVALUATION: role freeze, LONG + 3-view predictions, hard/prob aggregation, fixed ordinary ensembles, parent matching, metrics/statistics/burden and evidence export

**Gate:** `G18`  
**Inputs:** A4 R2 + matched model outputs  
**Outputs:** EnsembleControlRecord + A4 evidence

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("18")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G18"},indent=2))

## Stage 18U — Same-notebook additional full-execution ablation dispatcher; current extra set empty

**Gate:** `G18U`  
**Inputs:** Stage05 frozen unlock matrix  
**Outputs:** executed extra ablation records or NOT_AUTHORIZED rows

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("18U")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G18U"},indent=2))

## Stage 19 — Failure/missingness/negative-result accounting

**Gate:** `G19`  
**Inputs:** all branch ledgers  
**Outputs:** FailureCaseIndex + NegativeResultNote + diagnostics

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("19")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G19"},indent=2))

## Stage 20 — Downstream-readiness validation

**Gate:** `G20`  
**Inputs:** all records  
**Outputs:** Layer2ReadinessReport

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("20")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G20"},indent=2))

## Stage 21 — Figure-source and table-source datasets

**Gate:** `G21`  
**Inputs:** validated evidence  
**Outputs:** analysis/L10 source tables

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("21")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G21"},indent=2))

## Stage 22 — Protocol/Analysis/Layer0/EvidenceMap/Layer10/P03 handoffs

**Gate:** `G22`  
**Inputs:** validated bundle  
**Outputs:** governed handoff packets

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("22")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G22"},indent=2))

## Stage 23 — Gate matrix and evidence-sufficiency evaluation

**Gate:** `G23`  
**Inputs:** all validation  
**Outputs:** gate_decision + insufficiency route

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("23")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G23"},indent=2))

## Stage 24 — Final execution bundle/checksums/secret scan/export

**Gate:** `G24`  
**Inputs:** all outputs  
**Outputs:** immutable P02 bundle

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [ ]:
result = SESSION.run("24")
print(json.dumps({"stage_id":result["stage_id"],"status":result["status"],"elapsed_seconds":round(result["elapsed_seconds"],3),"gate":"G24"},indent=2))
# Stable Kaggle-output copy for retrieval; this is the same checksummed bundle, not a recomputation.
final_zip=Path(SESSION.finalization["zip"]["path"])
export_zip=KAGGLE_WORKING/final_zip.name
if final_zip.resolve()!=export_zip.resolve():
    shutil.copy2(final_zip,export_zip)
print(json.dumps({"final_runtime_bundle":str(export_zip),"run_id":SESSION.finalization["run_id"],"scientific_evidence":SESSION.finalization["scientific_evidence"]},indent=2))


## Completion boundary

If Stage 24 succeeds during the future real Kaggle run, the checksummed runtime bundle is placed under Kaggle's working/output area for retrieval. **Notebook success does not itself approve claims, perform Layer 0 review, or create final Evidence Map/Layer 10 interpretation.** Those remain downstream governed steps consuming this bundle.
